# Combined Feature Extraction Pipeline (ANOVA-Selected)

Builds merged ANOVA-selected features only. It avoids freezing by reading only selected columns from each family CSV.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd


def find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / 'pyproject.toml').exists():
            return path
    raise FileNotFoundError('Could not locate repository root (missing pyproject.toml).')


REPO_ROOT = find_repo_root(Path.cwd())
PIPELINE_DIR = REPO_ROOT / 'feature_extraction' / 'pipelines'
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

from pipeline_common import (
    ANOVA_FAMILIES,
    DEFAULT_ALL_FAMILIES,
    METADATA_COLUMNS,
    feature_columns_from_families,
    load_base_metadata,
    load_family_frames,
    load_selected_feature_lists,
    merge_feature_families,
    run_missing_family_generators,
)

OUT_DIR = REPO_ROOT / 'extracted_features' / 'combined'
OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
FAMILIES = ANOVA_FAMILIES.copy()
AUTO_RUN_MISSING = True
NOTEBOOK_TIMEOUT_SECONDS = 7200  # fail fast instead of hanging forever

selected_by_family = load_selected_feature_lists(REPO_ROOT)
include_features_by_family = {
    family: selected_by_family.get(family, [])
    for family in FAMILIES
}

if AUTO_RUN_MISSING:
    gen_results = run_missing_family_generators(
        REPO_ROOT,
        families=FAMILIES,
        execute_timeout=NOTEBOOK_TIMEOUT_SECONDS,
    )
    if gen_results:
        display(pd.DataFrame(gen_results))


In [ ]:
base_df = load_base_metadata(REPO_ROOT, include_xxx=False, require_agreement=True)
family_frames, family_report = load_family_frames(
    REPO_ROOT,
    families=FAMILIES,
    prefix_features=True,
    include_features_by_family=include_features_by_family,
)

report_df = pd.DataFrame(family_report).sort_values(['available', 'family'], ascending=[False, True])
display(report_df)

merged_df = merge_feature_families(base_df, family_frames)

selected_cols = []
missing_by_family = {}
for family in FAMILIES:
    missing = []
    for feature_name in selected_by_family.get(family, []):
        col = f"{family}__{feature_name}"
        if col in merged_df.columns:
            selected_cols.append(col)
        else:
            missing.append(col)
    if missing:
        missing_by_family[family] = missing

if missing_by_family:
    print('Missing selected columns by family:')
    for family, cols in missing_by_family.items():
        print(f'  {family}: {len(cols):,}')

meta_cols = [c for c in METADATA_COLUMNS if c in merged_df.columns]
narrow_df = merged_df[[*meta_cols, *selected_cols]].copy()

print(f'Base rows: {len(base_df):,}')
print(f'Loaded families: {len(family_frames):,} / {len(FAMILIES):,}')
print(f'Selected feature columns found: {len(selected_cols):,}')
print(f'Narrow dataset shape: {narrow_df.shape}')
narrow_df.head(2)


In [ ]:
OUT_CSV = OUT_DIR / 'anova_selected_combined_features.csv'
narrow_df.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')
